# Обучение YOLO11-seg: Сегментация пайки волноводов

**Авторы:** Даня, Боря, Алёна  
**Задача:** Сегментация 3 классов (waveguide, flux, solder) + определение этапов индукционной пайки  

## Инструкция:
1. Загрузи папку `data/dataset/` в Google Drive (в любое место, например `My Drive/НИР/data/dataset/`)
2. Убедись что внутри есть `data.yaml`, `images/train`, `images/val`, `images/test`, `labels/train`, `labels/val`, `labels/test`
3. Запускай ячейки по порядку
4. После обучения скачай `best.pt` и положи в `ДаняБоряНир/runs/yolo11n-seg/weights/`

## 1. Проверка GPU и установка зависимостей

In [ ]:
# Проверяем GPU
!nvidia-smi
import torch
print(f"\nPyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")

In [ ]:
# Установка ultralytics
!pip install -q ultralytics

## 2. Подключение Google Drive и настройка путей

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

# ===== НАСТРОЙ ЭТОТ ПУТЬ =====
# Укажи где на Google Drive лежит папка dataset
# Например: /content/drive/MyDrive/НИР/data/dataset
DRIVE_DATASET = "/content/drive/MyDrive/НИР/data/dataset"
# ==============================

# Проверяем что путь правильный
assert os.path.exists(DRIVE_DATASET), f"Папка не найдена: {DRIVE_DATASET}\nПроверь путь!"
assert os.path.exists(os.path.join(DRIVE_DATASET, 'data.yaml')), "data.yaml не найден!"
assert os.path.isdir(os.path.join(DRIVE_DATASET, 'images', 'train')), "images/train не найден!"

# Считаем файлы
for split in ['train', 'val', 'test']:
    img_dir = os.path.join(DRIVE_DATASET, 'images', split)
    lbl_dir = os.path.join(DRIVE_DATASET, 'labels', split)
    n_img = len(os.listdir(img_dir)) if os.path.isdir(img_dir) else 0
    n_lbl = len(os.listdir(lbl_dir)) if os.path.isdir(lbl_dir) else 0
    print(f"  {split}: {n_img} изображений, {n_lbl} аннотаций")

print("\nДатасет OK!")

In [ ]:
import yaml

# Копируем датасет на локальный диск Colab (быстрее чем с Drive)
LOCAL_DATASET = "/content/dataset"

if not os.path.exists(LOCAL_DATASET):
    print("Копирование датасета на локальный диск (1 раз)...")
    !cp -r "{DRIVE_DATASET}" "{LOCAL_DATASET}"
    print("Готово!")
else:
    print("Датасет уже скопирован")

# Исправляем path в data.yaml
data_yaml_path = os.path.join(LOCAL_DATASET, 'data.yaml')
with open(data_yaml_path, 'r') as f:
    data_cfg = yaml.safe_load(f)

data_cfg['path'] = LOCAL_DATASET

with open(data_yaml_path, 'w') as f:
    yaml.dump(data_cfg, f, default_flow_style=False, allow_unicode=True)

print(f"data.yaml path: {data_cfg['path']}")
print(f"Классы: {data_cfg['names']}")
print(f"Кол-во классов: {data_cfg['nc']}")

## 3. Обучение моделей

Обучаем 3 модели: nano, small, medium.  
Начинаем с nano (самая быстрая), потом можно small и medium.

In [ ]:
from ultralytics import YOLO

# ===== ПАРАМЕТРЫ ОБУЧЕНИЯ =====
EPOCHS = 150
PATIENCE = 30  # early stopping — остановится если 30 эпох нет улучшений

TRAIN_PARAMS = {
    'imgsz': 640,
    'patience': PATIENCE,
    'lr0': 0.01,
    'lrf': 0.01,
    'weight_decay': 0.0005,
    'warmup_epochs': 5,
    'mosaic': 1.0,
    'mixup': 0.15,
    'hsv_h': 0.015,
    'hsv_s': 0.7,
    'hsv_v': 0.4,
    'flipud': 0.5,
    'fliplr': 0.5,
    'degrees': 10.0,
    'translate': 0.1,
    'scale': 0.5,
    'workers': 2,
    'exist_ok': True,
    'verbose': True,
    'save': True,
    'save_period': 25,
}
# ==============================

print(f"Эпохи: {EPOCHS}, Early stopping: {PATIENCE}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

### 3.1 YOLO11n-seg (nano) — ~15 мин на T4

In [ ]:
# === NANO ===
model_nano = YOLO('yolo11n-seg.pt')

results_nano = model_nano.train(
    data=data_yaml_path,
    epochs=EPOCHS,
    batch=32,
    project='/content/runs',
    name='yolo11n-seg',
    device=0,
    **TRAIN_PARAMS,
)

print("\n=== NANO: обучение завершено ===")

### 3.2 YOLO11s-seg (small) — ~25 мин на T4

In [ ]:
# === SMALL ===
model_small = YOLO('yolo11s-seg.pt')

results_small = model_small.train(
    data=data_yaml_path,
    epochs=EPOCHS,
    batch=16,
    project='/content/runs',
    name='yolo11s-seg',
    device=0,
    **TRAIN_PARAMS,
)

print("\n=== SMALL: обучение завершено ===")

### 3.3 YOLO11m-seg (medium) — ~45 мин на T4

In [ ]:
# === MEDIUM ===
model_medium = YOLO('yolo11m-seg.pt')

results_medium = model_medium.train(
    data=data_yaml_path,
    epochs=EPOCHS,
    batch=8,
    project='/content/runs',
    name='yolo11m-seg',
    device=0,
    **TRAIN_PARAMS,
)

print("\n=== MEDIUM: обучение завершено ===")

## 4. Оценка на тестовой выборке

In [ ]:
import json

CLASSES = {0: 'waveguide', 1: 'flux', 2: 'solder'}
models_to_eval = ['yolo11n-seg', 'yolo11s-seg', 'yolo11m-seg']
all_results = []

for model_name in models_to_eval:
    best_path = f'/content/runs/{model_name}/weights/best.pt'
    if not os.path.exists(best_path):
        print(f"Пропуск {model_name} (не обучена)")
        continue

    print(f"\n{'='*50}")
    print(f"  Оценка: {model_name}")
    print(f"{'='*50}")

    model = YOLO(best_path)
    metrics = model.val(data=data_yaml_path, split='test', verbose=True)

    result = {
        'model': model_name,
        'seg_precision': float(metrics.seg.mp),
        'seg_recall': float(metrics.seg.mr),
        'seg_mAP50': float(metrics.seg.map50),
        'seg_mAP50_95': float(metrics.seg.map),
        'box_mAP50': float(metrics.box.map50),
        'speed_inference_ms': float(metrics.speed['inference']),
    }

    for i, name in CLASSES.items():
        if i < len(metrics.seg.p):
            result[f'precision_{name}'] = float(metrics.seg.p[i])
            result[f'recall_{name}'] = float(metrics.seg.r[i])
            result[f'ap50_{name}'] = float(metrics.seg.ap50[i])

    all_results.append(result)
    print(f"  mAP50(seg): {result['seg_mAP50']:.4f}")
    print(f"  mAP50-95(seg): {result['seg_mAP50_95']:.4f}")

# Сравнительная таблица
if all_results:
    print(f"\n{'='*70}")
    print("  СРАВНЕНИЕ МОДЕЛЕЙ")
    print(f"{'='*70}")
    print(f"{'Модель':<18} {'Prec':>8} {'Recall':>8} {'mAP50':>8} {'mAP50-95':>10} {'ms':>8}")
    print('-' * 60)
    for r in all_results:
        print(f"{r['model']:<18} {r['seg_precision']:>8.4f} {r['seg_recall']:>8.4f} "
              f"{r['seg_mAP50']:>8.4f} {r['seg_mAP50_95']:>10.4f} {r['speed_inference_ms']:>8.1f}")

    best = max(all_results, key=lambda x: x['seg_mAP50_95'])
    print(f"\n  Лучшая: {best['model']} (mAP50-95 = {best['seg_mAP50_95']:.4f})")

    # Сохраняем
    with open('/content/runs/comparison.json', 'w') as f:
        json.dump(all_results, f, indent=2, ensure_ascii=False)

## 5. Визуализация предсказаний

In [ ]:
import matplotlib.pyplot as plt
import cv2
import numpy as np

# Берём лучшую модель
best_model_name = max(all_results, key=lambda x: x['seg_mAP50_95'])['model']
best_path = f'/content/runs/{best_model_name}/weights/best.pt'
model = YOLO(best_path)

# Тестовые изображения
test_dir = os.path.join(LOCAL_DATASET, 'images', 'test')
test_images = sorted([os.path.join(test_dir, f) for f in os.listdir(test_dir)
                       if f.lower().endswith(('.jpg', '.png'))])[:8]

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
fig.suptitle(f'Предсказания {best_model_name}', fontsize=16)

for ax, img_path in zip(axes.flat, test_images):
    results = model.predict(img_path, conf=0.5, verbose=False)
    annotated = results[0].plot()
    annotated = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)
    ax.imshow(annotated)
    ax.set_title(os.path.basename(img_path)[:25], fontsize=9)
    ax.axis('off')

plt.tight_layout()
plt.savefig('/content/runs/predictions.png', dpi=150, bbox_inches='tight')
plt.show()
print("Сохранено: /content/runs/predictions.png")

## 6. Графики обучения

In [ ]:
import pandas as pd

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for model_name in models_to_eval:
    csv_path = f'/content/runs/{model_name}/results.csv'
    if not os.path.exists(csv_path):
        continue
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()

    label = model_name.replace('yolo11', '').replace('-seg', '')

    # mAP50 seg
    if 'metrics/mAP50(M)' in df.columns:
        axes[0].plot(df['epoch'], df['metrics/mAP50(M)'], label=label)
    # seg loss
    if 'train/seg_loss' in df.columns:
        axes[1].plot(df['epoch'], df['train/seg_loss'], label=label)
    # box loss
    if 'train/box_loss' in df.columns:
        axes[2].plot(df['epoch'], df['train/box_loss'], label=label)

axes[0].set_title('mAP50 (Seg)'); axes[0].set_xlabel('Epoch'); axes[0].legend()
axes[1].set_title('Seg Loss'); axes[1].set_xlabel('Epoch'); axes[1].legend()
axes[2].set_title('Box Loss'); axes[2].set_xlabel('Epoch'); axes[2].legend()

plt.tight_layout()
plt.savefig('/content/runs/training_curves.png', dpi=150)
plt.show()

## 7. Per-class метрики лучшей модели

In [ ]:
best_result = max(all_results, key=lambda x: x['seg_mAP50_95'])

print(f"Лучшая модель: {best_result['model']}")
print(f"\n{'Класс':<12} {'Precision':>10} {'Recall':>10} {'AP50':>10}")
print('-' * 45)
for name in ['waveguide', 'flux', 'solder']:
    p = best_result.get(f'precision_{name}', 0)
    r = best_result.get(f'recall_{name}', 0)
    ap = best_result.get(f'ap50_{name}', 0)
    print(f"{name:<12} {p:>10.4f} {r:>10.4f} {ap:>10.4f}")

print(f"\n{'ВСЕГО':<12} {best_result['seg_precision']:>10.4f} {best_result['seg_recall']:>10.4f} {best_result['seg_mAP50']:>10.4f}")
print(f"\nmAP50-95: {best_result['seg_mAP50_95']:.4f}")
print(f"Скорость: {best_result['speed_inference_ms']:.1f} ms")

## 8. Скачивание весов

Скачай `best.pt` лучшей модели и положи в `ДаняБоряНир/runs/<model_name>/weights/best.pt`

In [ ]:
# Копируем на Google Drive для сохранности
DRIVE_SAVE = "/content/drive/MyDrive/НИР/trained_weights"
os.makedirs(DRIVE_SAVE, exist_ok=True)

for model_name in models_to_eval:
    best_pt = f'/content/runs/{model_name}/weights/best.pt'
    if os.path.exists(best_pt):
        dest = os.path.join(DRIVE_SAVE, f'{model_name}_best.pt')
        !cp "{best_pt}" "{dest}"
        size_mb = os.path.getsize(dest) / 1024**2
        print(f"  {model_name}: сохранено ({size_mb:.1f} MB)")

# Также копируем графики и результаты
!cp /content/runs/comparison.json "{DRIVE_SAVE}/" 2>/dev/null
!cp /content/runs/predictions.png "{DRIVE_SAVE}/" 2>/dev/null
!cp /content/runs/training_curves.png "{DRIVE_SAVE}/" 2>/dev/null

print(f"\nВсё сохранено в: {DRIVE_SAVE}")
print("Скачай best.pt и положи в ДаняБоряНир/runs/<model>/weights/best.pt")

In [ ]:
# Или скачай напрямую в браузер
from google.colab import files

best_model_name = max(all_results, key=lambda x: x['seg_mAP50_95'])['model']
best_pt = f'/content/runs/{best_model_name}/weights/best.pt'
print(f"Скачиваю {best_model_name}/best.pt...")
files.download(best_pt)